# Module 2 — Production-Grade Prompting, Agents & Tool-use

Code-first offline drills. Start with the [29-screen module index](../course%20content%20HTML/02-production-grade-prompting-agents-tool-use/index.html#complete-lesson-index). Optional live calls use the one project-root `.env`.

In [ ]:
from pathlib import Path
import json
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'study_support.py').is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from study_support import load_anthropic_api_key, messages_create

assert callable(load_anthropic_api_key) and callable(messages_create)

## Prompt contracts and malformed-output retry

Diagnose the missing structure, then add only that structure: [archive § S02](../course%20content%20HTML/02-production-grade-prompting-agents-tool-use/02-prompting-craft.html#s02--teaching) · [course S02](https://anthropic-partners.skilljar.com/content/wp/4hdejjwplbrm/3fpv056zo4wp4/Developer_M2_vF2.html#S02).

In [ ]:
def ticket_request(ticket: str) -> dict:
    return {
        'system': 'Classify tickets. Return only JSON with category and urgency.',
        'messages': [{'role': 'user', 'content': f'<ticket>{ticket}</ticket>'}],
        'output_config': {'format': {'type': 'json_schema', 'schema': {
            'type': 'object',
            'properties': {'category': {'enum': ['billing', 'technical', 'escalation']},
                           'urgency': {'enum': ['low', 'medium', 'high', 'critical']}},
            'required': ['category', 'urgency'], 'additionalProperties': False}}},
    }

def parse_or_retry(raw: str) -> dict:
    try:
        value = json.loads(raw)
        if set(value) != {'category', 'urgency'}:
            raise ValueError('wrong fields')
        return {'ok': True, 'value': value}
    except (json.JSONDecodeError, ValueError) as error:
        return {'ok': False, 'retry': f'Return the same answer as schema-valid JSON. Error: {error}'}

assert parse_or_retry('{bad json')['ok'] is False
assert parse_or_retry('{"category":"technical","urgency":"high"}')['ok'] is True
assert ticket_request('API key failed')['output_config']['format']['type'] == 'json_schema'

## Adaptive thinking and carry-back

Enable adaptive reasoning for dependent planning, tune effort, and preserve thinking blocks unchanged: [archive § S05](../course%20content%20HTML/02-production-grade-prompting-agents-tool-use/03-extended-thinking.html#s05--teaching) · [course S05](https://anthropic-partners.skilljar.com/content/wp/4hdejjwplbrm/3fpv056zo4wp4/Developer_M2_vF2.html#S05).

In [ ]:
def reasoning_controls(task: str, effort: str = 'medium') -> dict:
    mechanical = {'classify', 'extract', 'format'}
    return {} if task in mechanical else {'thinking': {'type': 'adaptive'}, 'output_config': {'effort': effort}}

def continue_after_tools(assistant_content: list[dict], results: list[dict]) -> list[dict]:
    return [
        {'role': 'assistant', 'content': assistant_content},
        {'role': 'user', 'content': results},
    ]

thinking = {'type': 'thinking', 'thinking': 'plan', 'signature': 'signed'}
turns = continue_after_tools([thinking, {'type': 'tool_use', 'id': 't1', 'name': 'lookup', 'input': {}}],
                             [{'type': 'tool_result', 'tool_use_id': 't1', 'content': 'done'}])
assert reasoning_controls('classify') == {}
assert reasoning_controls('plan', 'high')['thinking'] == {'type': 'adaptive'}
assert turns[0]['content'][0] is thinking

## Tool schemas, pairing, and parallel calls

Descriptions need inclusion and exclusion boundaries; every `tool_use` ID needs an immediate matching result: [archive § S07](../course%20content%20HTML/02-production-grade-prompting-agents-tool-use/04-tool-use-and-schema-design.html#s07--teaching) · [course S07](https://anthropic-partners.skilljar.com/content/wp/4hdejjwplbrm/3fpv056zo4wp4/Developer_M2_vF2.html#S07).

In [ ]:
TOOLS = [{
    'name': 'get_account_balance',
    'description': 'Get the current balance for one account ID. Do not use for transaction history.',
    'input_schema': {'type': 'object', 'properties': {'account_id': {'type': 'string'}},
                     'required': ['account_id'], 'additionalProperties': False},
}]

def tool_results(calls: list[dict], execute) -> list[dict]:
    return [{'type': 'tool_result', 'tool_use_id': call['id'], 'content': execute(call)} for call in calls]

def valid_pairing(calls: list[dict], results: list[dict]) -> bool:
    return {c['id'] for c in calls} == {r['tool_use_id'] for r in results}

calls = [{'id': 't1', 'name': 'get_account_balance', 'input': {'account_id': 'A-1'}},
         {'id': 't2', 'name': 'get_account_balance', 'input': {'account_id': 'A-2'}}]
results = tool_results(calls, lambda c: f"balance:{c['input']['account_id']}")
assert valid_pairing(calls, results) and len(results) == 2
assert not valid_pairing(calls, results[:-1])

## Stream recovery

A read loop ending is not completion. Commit only after `message_stop`; discard partial blocks and retry from the last complete turn: [archive § S10](../course%20content%20HTML/02-production-grade-prompting-agents-tool-use/05-streaming-responses.html#s10--teaching) · [course S10](https://anthropic-partners.skilljar.com/content/wp/4hdejjwplbrm/3fpv056zo4wp4/Developer_M2_vF2.html#S10).

In [ ]:
def assemble_stream(events: list[dict]) -> list[dict]:
    blocks, closed, stopped = {}, set(), False
    for event in events:
        index = event.get('index')
        if event['type'] == 'content_block_start':
            blocks[index] = {**event['content_block'], 'text': ''}
        elif event['type'] == 'content_block_delta':
            blocks[index]['text'] += event['delta'].get('text', '')
        elif event['type'] == 'content_block_stop':
            closed.add(index)
        elif event['type'] == 'message_stop':
            stopped = True
    if not stopped or closed != set(blocks):
        raise InterruptedError('discard partial turn; retry from last complete turn')
    return [blocks[i] for i in sorted(blocks)]

complete = [
    {'type': 'content_block_start', 'index': 0, 'content_block': {'type': 'text'}},
    {'type': 'content_block_delta', 'index': 0, 'delta': {'text': 'done'}},
    {'type': 'content_block_stop', 'index': 0}, {'type': 'message_stop'},
]
assert assemble_stream(complete)[0]['text'] == 'done'
try:
    assemble_stream(complete[:-1])
    raise AssertionError('partial stream accepted')
except InterruptedError:
    pass

## Context budget and prompt-cache checkpoints

Cache stable prefixes, keep at most four breakpoints, and compact before the budget fails: [archive § S13](../course%20content%20HTML/02-production-grade-prompting-agents-tool-use/06-context-engineering.html#s13--teaching) · [course S13](https://anthropic-partners.skilljar.com/content/wp/4hdejjwplbrm/3fpv056zo4wp4/Developer_M2_vF2.html#S13).

In [ ]:
def cache_prefix(blocks: list[dict], checkpoints: tuple[int, ...]) -> list[dict]:
    if len(checkpoints) > 4:
        raise ValueError('at most four cache breakpoints')
    marked = [dict(block) for block in blocks]
    for index in checkpoints:
        marked[index]['cache_control'] = {'type': 'ephemeral'}
    return marked

def context_action(tokens: int, budget: int) -> str:
    ratio = tokens / budget
    return 'compact' if ratio >= .8 else 'measure' if ratio >= .6 else 'continue'

prefix = cache_prefix([{'type': 'text', 'text': 'stable system prompt'},
                       {'type': 'text', 'text': 'reference corpus'}], (1,))
assert prefix[1]['cache_control'] == {'type': 'ephemeral'}
assert context_action(33_000, 40_000) == 'compact'

## Choose workflow, custom loop, Agent SDK, framework, or managed runtime

The agent pattern stays constant; choose who owns the loop based on the constraints: [archive § S16](../course%20content%20HTML/02-production-grade-prompting-agents-tool-use/07-agent-construction.html#s16--teaching) · [course S16](https://anthropic-partners.skilljar.com/content/wp/4hdejjwplbrm/3fpv056zo4wp4/Developer_M2_vF2.html#S16).

In [ ]:
def choose_runtime(*, fixed_steps=False, full_control=False, long_running=False,
                   existing_framework=False, regulated=False) -> str:
    if fixed_steps:
        return 'workflow'
    if existing_framework:
        return 'existing framework'
    if long_running and not regulated:
        return 'Claude Managed Agents'
    if full_control or regulated:
        return 'custom Messages API loop'
    return 'Claude Agent SDK'

assert choose_runtime(fixed_steps=True) == 'workflow'
assert choose_runtime(full_control=True) == 'custom Messages API loop'
assert choose_runtime() == 'Claude Agent SDK'
assert choose_runtime(existing_framework=True) == 'existing framework'
assert choose_runtime(long_running=True) == 'Claude Managed Agents'

## Memory scope and HITL

Persist only state that must cross sessions, and gate irreversible tools before execution: [archive § S19](../course%20content%20HTML/02-production-grade-prompting-agents-tool-use/08-agent-memory.html#s19--teaching) · [course S19](https://anthropic-partners.skilljar.com/content/wp/4hdejjwplbrm/3fpv056zo4wp4/Developer_M2_vF2.html#S19) · [archive § S18](../course%20content%20HTML/02-production-grade-prompting-agents-tool-use/07-agent-construction.html#s18--checkpoint) · [course S18](https://anthropic-partners.skilljar.com/content/wp/4hdejjwplbrm/3fpv056zo4wp4/Developer_M2_vF2.html#S18).

In [ ]:
def memory_scope(*, crosses_sessions: bool, long_session: bool) -> str:
    if crosses_sessions:
        return 'external storage'
    return 'summarized' if long_session else 'in-context'

def execute_with_hitl(call: dict, execute, approve) -> dict:
    if call['name'].startswith(('write_', 'delete_', 'send_')) and not approve(call):
        return {'type': 'tool_result', 'tool_use_id': call['id'], 'content': 'Rejected by operator'}
    return {'type': 'tool_result', 'tool_use_id': call['id'], 'content': execute(call)}

assert memory_scope(crosses_sessions=True, long_session=False) == 'external storage'
rejected = execute_with_hitl({'id': 't3', 'name': 'write_record'}, lambda _: 'changed', lambda _: False)
assert rejected['content'] == 'Rejected by operator'

## Vision, PDFs, Files API, and Message Batches

Use inline data for one-offs, `file_id` for reuse, `document` for PDFs, and batches for offline volume: [archive § S24](../course%20content%20HTML/02-production-grade-prompting-agents-tool-use/10-multimodal-and-batch-ingestion.html#s24--teaching) · [course S24](https://anthropic-partners.skilljar.com/content/wp/4hdejjwplbrm/3fpv056zo4wp4/Developer_M2_vF2.html#S24).

In [ ]:
def visual_tokens(width: int, height: int) -> int:
    return ((width + 27) // 28) * ((height + 27) // 28)

def reusable_file(file_id: str, *, pdf=False) -> dict:
    return {'type': 'document' if pdf else 'image', 'source': {'type': 'file', 'file_id': file_id}}

def batch_requests(items: list[str]) -> list[dict]:
    return [{'custom_id': f'item-{i}', 'params': {'messages': [{'role': 'user', 'content': item}]}}
            for i, item in enumerate(items)]

assert visual_tokens(1000, 1000) == 1296
assert reusable_file('file_1', pdf=True)['type'] == 'document'
assert [r['custom_id'] for r in batch_requests(['a', 'b'])] == ['item-0', 'item-1']

## Exercise — harden one production path

Use the four planted failure layers from [archive § S22](../course%20content%20HTML/02-production-grade-prompting-agents-tool-use/09-cumulative-debug-task.html#s22--identify) · [course S22](https://anthropic-partners.skilljar.com/content/wp/4hdejjwplbrm/3fpv056zo4wp4/Developer_M2_vF2.html#S22). Add one failing production case, choose its layer, then add the smallest guard that makes the assertion pass.

In [ ]:
FAILURE_LAYER = {
    'wrong_tool': 'schema',
    'partial_turn_committed': 'streaming',
    'orphan_tool_result': 'context',
    'session_four_overflow': 'memory',
}

def guard_for(symptom: str) -> str:
    return {
        'schema': 'add an exclusion condition',
        'streaming': 'commit only after message_stop',
        'context': 'preserve the complete assistant tool_use turn',
        'memory': 'externalize prior sessions and inject only relevant state',
    }[FAILURE_LAYER[symptom]]

assert guard_for('partial_turn_committed') == 'commit only after message_stop'
# Exercise: add a fifth symptom, its layer, and one exact assertion.